# DS605 Challenge: Black-Box Hyperparameter Optimization

### Objective
Find the hyperparameter configuration that gives the minimum loss
using as few Oracle calls as possible.

### Constraints
- No automated hyperparameter optimization libraries
- Only assigned hyperparameter values may be used
- Every Oracle call counts
- Lower loss is better
- Avoid repeated configurations

In [2]:
import json
from urllib.request import Request, urlopen

TEAM_ID = "TEAM_04"
API_KEY = "oc_Q9MtIL5yj8lb3h3VgAL98xLSW5XEnuQ_"
API_URL = "https://bftrxasgtunepckchcoz.supabase.co/functions/v1/oracle"


def api(payload):
    req = Request(
        API_URL,
        data=json.dumps(payload).encode(),
        headers={
            "Content-Type": "application/json",
            "X-API-Key": API_KEY
        }
    )

    with urlopen(req, timeout=60) as r:
        return json.load(r)

In [3]:
# Check assigned model and hyperparameter search space

spec = api({
    "action": "spec",
    "team_id": TEAM_ID
})

SPACE = spec["hyperparameters"]

print("Model:", spec["model"])
print("\nAssigned Hyperparameter Search Space:")

for name, values in SPACE.items():
    print(name, ":", values)

Model: DecisionTreeRegressor

Assigned Hyperparameter Search Space:
splitter : ['best', 'random']
criterion : ['squared_error', 'friedman_mse']
max_depth : [2, 3, 4, 5, 6, 8, 10, None]
max_features : [0.4, 0.6, 0.8, 1]
min_samples_leaf : [1, 2, 4, 8]
min_samples_split : [2, 5, 10, 20]


In [4]:
def oracle_query(params):
    result = api({
        "action": "query",
        "team_id": TEAM_ID,
        "params": params
    })
    return float(result["loss"])

In [5]:
history = []
best_params = None
best_loss = float("inf")
oracle_calls = 0

In [6]:
def evaluate(params):
    global best_params, best_loss, oracle_calls

    loss = oracle_query(params)
    oracle_calls += 1

    history.append({
        "params": params.copy(),
        "loss": loss
    })

    if loss < best_loss:
        best_loss = loss
        best_params = params.copy()

    print("Call:", oracle_calls)
    print("Params:", params)
    print("Loss:", loss)
    print("Best loss so far:", best_loss)
    print("-" * 50)

In [7]:
baseline = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 5,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(baseline)

Call: 1
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 5, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 10.5579568771188
Best loss so far: 10.5579568771188
--------------------------------------------------


In [8]:
print("Successful Oracle calls recorded:", oracle_calls)
print("Configurations recorded:", len(history))

for item in history:
    print(item)

Successful Oracle calls recorded: 1
Configurations recorded: 1
{'params': {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 5, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}, 'loss': 10.5579568771188}


In [9]:
# Test configuration: max_depth = 3

params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 3,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 2
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 3, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 12.5631605577281
Best loss so far: 10.5579568771188
--------------------------------------------------


In [10]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 3
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 9.40723856248771
Best loss so far: 9.40723856248771
--------------------------------------------------


In [11]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 8,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 4
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 8, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 13.7232750740626
Best loss so far: 9.40723856248771
--------------------------------------------------


In [12]:
# Phase 1: Explore a larger tree depth
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 8,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 5
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 8, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 13.7232750740626
Best loss so far: 9.40723856248771
--------------------------------------------------


In [13]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 1,
    "min_samples_split": 5
}

evaluate(params)

Call: 6
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 1, 'min_samples_split': 5}
Loss: 9.55710762849283
Best loss so far: 9.40723856248771
--------------------------------------------------


In [14]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.6,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 7
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.6, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 12.0284165785446
Best loss so far: 9.40723856248771
--------------------------------------------------


In [15]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 1.0,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 8
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 1.0, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 9.96889435488513
Best loss so far: 9.40723856248771
--------------------------------------------------


In [16]:
params = {
    "splitter": "best",
    "criterion": "friedman_mse",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 9
Params: {'splitter': 'best', 'criterion': 'friedman_mse', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 9.40723856248771
Best loss so far: 9.40723856248771
--------------------------------------------------


In [17]:
params = {
    "splitter": "random",
    "criterion": "friedman_mse",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 10
Params: {'splitter': 'random', 'criterion': 'friedman_mse', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 12.9558686219841
Best loss so far: 9.40723856248771
--------------------------------------------------


In [18]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 10
}

evaluate(params)

Call: 11
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 10}
Loss: 13.9557689323983
Best loss so far: 9.40723856248771
--------------------------------------------------


In [19]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 2
}

evaluate(params)

Call: 12
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 2}
Loss: 9.40723856248771
Best loss so far: 9.40723856248771
--------------------------------------------------


In [20]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 4,
    "min_samples_split": 5
}

evaluate(params)

Call: 13
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 4, 'min_samples_split': 5}
Loss: 10.4314202251831
Best loss so far: 9.40723856248771
--------------------------------------------------


In [21]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 4,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 14
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 4, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 12.3626301986368
Best loss so far: 9.40723856248771
--------------------------------------------------


In [22]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 4,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 15
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 4, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 12.3626301986368
Best loss so far: 9.40723856248771
--------------------------------------------------


In [23]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.4,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 16
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.4, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 12.1399871162503
Best loss so far: 9.40723856248771
--------------------------------------------------


In [24]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 20
}

evaluate(params)

Call: 17
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 20}
Loss: 13.312439335618
Best loss so far: 9.40723856248771
--------------------------------------------------


In [25]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 10,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 18
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 10, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 14.0784755690227
Best loss so far: 9.40723856248771
--------------------------------------------------


In [26]:
params = {
    "splitter": "random",
    "criterion": "squared_error",
    "max_depth": 5,
    "max_features": 0.6,
    "min_samples_leaf": 4,
    "min_samples_split": 10
}

evaluate(params)

Call: 19
Params: {'splitter': 'random', 'criterion': 'squared_error', 'max_depth': 5, 'max_features': 0.6, 'min_samples_leaf': 4, 'min_samples_split': 10}
Loss: 12.0889006225343
Best loss so far: 9.40723856248771
--------------------------------------------------


In [27]:
params = {
    "splitter": "random",
    "criterion": "friedman_mse",
    "max_depth": None,
    "max_features": 0.4,
    "min_samples_leaf": 8,
    "min_samples_split": 20
}

evaluate(params)

Call: 20
Params: {'splitter': 'random', 'criterion': 'friedman_mse', 'max_depth': None, 'max_features': 0.4, 'min_samples_leaf': 8, 'min_samples_split': 20}
Loss: 14.0839772942953
Best loss so far: 9.40723856248771
--------------------------------------------------


In [28]:
params = {
    "splitter": "best",
    "criterion": "friedman_mse",
    "max_depth": 3,
    "max_features": 0.4,
    "min_samples_leaf": 8,
    "min_samples_split": 20
}

evaluate(params)

Call: 21
Params: {'splitter': 'best', 'criterion': 'friedman_mse', 'max_depth': 3, 'max_features': 0.4, 'min_samples_leaf': 8, 'min_samples_split': 20}
Loss: 15.8704278797868
Best loss so far: 9.40723856248771
--------------------------------------------------


In [29]:
params = {
    "splitter": "random",
    "criterion": "squared_error",
    "max_depth": 8,
    "max_features": 1.0,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

evaluate(params)

Call: 22
Params: {'splitter': 'random', 'criterion': 'squared_error', 'max_depth': 8, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 2}
Loss: 13.1333205991435
Best loss so far: 9.40723856248771
--------------------------------------------------


In [30]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": None,
    "max_features": 0.6,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

evaluate(params)

Call: 23
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': None, 'max_features': 0.6, 'min_samples_leaf': 1, 'min_samples_split': 2}
Loss: 14.678269787544
Best loss so far: 9.40723856248771
--------------------------------------------------


In [31]:
params = {
    "splitter": "random",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 24
Params: {'splitter': 'random', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 12.9558686219841
Best loss so far: 9.40723856248771
--------------------------------------------------


In [32]:
params = {
    "splitter": "best",
    "criterion": "friedman_mse",
    "max_depth": 5,
    "max_features": 1.0,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

evaluate(params)

Call: 25
Params: {'splitter': 'best', 'criterion': 'friedman_mse', 'max_depth': 5, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 2}
Loss: 10.831448286938
Best loss so far: 9.40723856248771
--------------------------------------------------


In [33]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 2,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 5
}

evaluate(params)

Call: 26
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 2, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}
Loss: 14.999105251512
Best loss so far: 9.40723856248771
--------------------------------------------------


In [34]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 5,
    "max_features": 0.4,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

evaluate(params)

Call: 27
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 5, 'max_features': 0.4, 'min_samples_leaf': 1, 'min_samples_split': 2}
Loss: 11.553105500449
Best loss so far: 9.40723856248771
--------------------------------------------------


In [35]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 1.0,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

evaluate(params)

Call: 28
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 1.0, 'min_samples_leaf': 1, 'min_samples_split': 2}
Loss: 10.1167758650544
Best loss so far: 9.40723856248771
--------------------------------------------------


In [36]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

evaluate(params)

Call: 29
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 1, 'min_samples_split': 2}
Loss: 10.6954042256234
Best loss so far: 9.40723856248771
--------------------------------------------------


In [37]:
params = {
    "splitter": "best",
    "criterion": "friedman_mse",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

evaluate(params)

Call: 30
Params: {'splitter': 'best', 'criterion': 'friedman_mse', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 1, 'min_samples_split': 2}
Loss: 10.6954042256234
Best loss so far: 9.40723856248771
--------------------------------------------------


In [38]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 8,
    "min_samples_split": 5
}

evaluate(params)

Call: 31
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 8, 'min_samples_split': 5}
Loss: 12.2209583396949
Best loss so far: 9.40723856248771
--------------------------------------------------


In [39]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 1.0,
    "min_samples_leaf": 2,
    "min_samples_split": 2
}

evaluate(params)

Call: 32
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 1.0, 'min_samples_leaf': 2, 'min_samples_split': 2}
Loss: 9.96889435488513
Best loss so far: 9.40723856248771
--------------------------------------------------


In [40]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 5,
    "max_features": 0.6,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

evaluate(params)

Call: 33
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 5, 'max_features': 0.6, 'min_samples_leaf': 1, 'min_samples_split': 2}
Loss: 12.3637970582453
Best loss so far: 9.40723856248771
--------------------------------------------------


In [41]:
params = {
    "splitter": "best",
    "criterion": "friedman_mse",
    "max_depth": 6,
    "max_features": 0.8,
    "min_samples_leaf": 4,
    "min_samples_split": 2
}

evaluate(params)

Call: 34
Params: {'splitter': 'best', 'criterion': 'friedman_mse', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 4, 'min_samples_split': 2}
Loss: 10.4314202251831
Best loss so far: 9.40723856248771
--------------------------------------------------


In [42]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 5,
    "max_features": 0.8,
    "min_samples_leaf": 1,
    "min_samples_split": 2
}

evaluate(params)

Call: 35
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 5, 'max_features': 0.8, 'min_samples_leaf': 1, 'min_samples_split': 2}
Loss: 10.678841922379
Best loss so far: 9.40723856248771
--------------------------------------------------


In [43]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 6,
    "max_features": 0.6,
    "min_samples_leaf": 2,
    "min_samples_split": 2
}

evaluate(params)

Call: 36
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.6, 'min_samples_leaf': 2, 'min_samples_split': 2}
Loss: 11.0278001211652
Best loss so far: 9.40723856248771
--------------------------------------------------


In [44]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 5,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 2
}

evaluate(params)

Call: 37
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 5, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 2}
Loss: 10.5579568771188
Best loss so far: 9.40723856248771
--------------------------------------------------


In [45]:
params = {
    "splitter": "best",
    "criterion": "squared_error",
    "max_depth": 5,
    "max_features": 0.8,
    "min_samples_leaf": 2,
    "min_samples_split": 2
}

evaluate(params)

Call: 38
Params: {'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 5, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 2}
Loss: 10.5579568771188
Best loss so far: 9.40723856248771
--------------------------------------------------


In [46]:
# Final Best Configuration

print("Best Hyperparameters:")
print(best_params)

print("\nBest Loss:")
print(best_loss)

print("\nTotal Oracle Calls:")
print(oracle_calls)

Best Hyperparameters:
{'splitter': 'best', 'criterion': 'squared_error', 'max_depth': 6, 'max_features': 0.8, 'min_samples_leaf': 2, 'min_samples_split': 5}

Best Loss:
9.40723856248771

Total Oracle Calls:
38
